In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import os
import re
import matplotlib.pyplot as plt

from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV, train_test_split, RepeatedKFold
from sklearn.inspection import PartialDependenceDisplay
from scipy.stats import randint, uniform, loguniform

# 📁 文件夹路径
grid_folder = r'D:\seoul\grids\lst_map'

# 🔧 变量定义
target_vars = ['nor_2020', 'ext_2020', 'hr_2020']
explanatory_vars = ['BCR(%)', 'BHV', 'NDVI', 'SVF', 'EV(m)',
                    'Dist_BP', 'Dist_MT', 'Dist_WB', 'WR(%)']

# 📊 保存结果
all_results = []
pdp_records = []
r2_comparison = []

# 🔍 ANN 的随机搜索空间
param_dist = {
    # 隐藏层结构：[单层、双层、三层] × [每层节点数范围]
    'hidden_layer_sizes': [(h1,) for h1 in randint(50, 300).rvs(5)] +
                          [(h1, h2) for h1, h2 in zip(randint(50, 300).rvs(5), randint(20, 150).rvs(5))],
    # 激活函数
    'activation': ['tanh', 'relu', 'logistic'],
    # 学习率初始值
    'learning_rate_init': loguniform(1e-4, 1e-1),
    # L2 正则化参数
    'alpha': loguniform(1e-6, 1e-2),
    # 随机梯度下降或 Adam
    'solver': ['adam', 'lbfgs'],
    # 批量大小
    'batch_size': [32, 64, 128, 'auto'],
}

# === 主循环 ===
for filename in os.listdir(grid_folder):
    if not filename.endswith('_clean.shp'):
        continue

    input_path = os.path.join(grid_folder, filename)
    match = re.search(r'(\d{3,5})m', filename)
    grid_size = match.group(1)

    gdf = gpd.read_file(input_path)
    gdf_clean = gdf.replace([np.inf, -np.inf], np.nan).dropna(subset=target_vars + explanatory_vars)

    for target in target_vars:
        X = gdf_clean[explanatory_vars]
        y = gdf_clean[target]

        # 🔀 数据划分
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=0
        )

        # 🧠 定义 ANN 模型
        ann = MLPRegressor(
            random_state=0,
            max_iter=1000,
            early_stopping=True,
            n_iter_no_change=20,
            tol=1e-4,
        )

        # 🔄 随机搜索
        cv = RepeatedKFold(n_splits=5, n_repeats=4, random_state=0)
        search = RandomizedSearchCV(
            estimator=ann,
            param_distributions=param_dist,
            n_iter=100,          # 根据需要调节
            scoring='r2',
            cv=cv,
            verbose=2,
            n_jobs=-1,
            random_state=0
        )
        search.fit(X_train, y_train)

        # ✅ 用测试集评估
        best_model = search.best_estimator_
        y_train_pred = best_model.predict(X_train)
        y_test_pred  = best_model.predict(X_test)

        r2_train = best_model.score(X_train, y_train)
        r2_test  = r2_score(y_test, y_test_pred)
        rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
        rmse_test  = np.sqrt(mean_squared_error(y_test, y_test_pred))

        print(f"✅ {filename} | {target} 最佳参数: {search.best_params_} "
              f"| R²_train={r2_train:.3f} | R²_test={r2_test:.3f}")

        # 记录结果
        for var, importance in zip(explanatory_vars, getattr(best_model, 'coefs_[0]', [])):
            # MLP 没有 feature_importances_，这里示例用第一层权重绝对值均值作为“重要性”
            fi = np.mean(np.abs(best_model.coefs_[0]), axis=1)[explanatory_vars.index(var)]
            all_results.append({
                'GridSize': grid_size,
                'Target': target,
                'Feature': var,
                'FeatureImportance_TrainModel': round(fi, 4),
                'Train_R2': round(r2_train, 4),
                'Train_RMSE': round(rmse_train, 4),
                'Test_R2': round(r2_test, 4),
                'Test_RMSE': round(rmse_test, 4),
                **search.best_params_
            })
        r2_comparison.append({
            'GridSize': grid_size,
            'Target': target,
            'Train_R2': round(r2_train, 4),
            'Test_R2': round(r2_test, 4),
            'Train_RMSE': round(rmse_train, 4),
            'Test_RMSE': round(rmse_test, 4)
        })

        # 📈 PDP 提取（Top 9 特征）
        # 注意：MLP 没有内建的 feature_importances_，这里用我们上面算出的 fi 作为 proxy
        fi_arr = np.array([r['FeatureImportance_TrainModel'] for r in all_results if r['GridSize']==grid_size and r['Target']==target])
        top_idx = np.argsort(fi_arr)[::-1][:9]
        top_features = [explanatory_vars[i] for i in top_idx]

        for feature in top_features:
            fig, ax = plt.subplots()
            disp = PartialDependenceDisplay.from_estimator(best_model, X, [feature], ax=ax)
            x_vals = disp.lines_[0][0].get_xdata()
            y_vals = disp.lines_[0][0].get_ydata()
            plt.close(fig)
            pdp_records.append({
                'Feature': feature,
                'GridSize': grid_size,
                'Target': target,
                'X': x_vals,
                'Y': y_vals
            })

# 保存结果
pd.DataFrame(all_results).to_excel(os.path.join(grid_folder, 'ANN_Random_Search_Results.xlsx'), index=False)
pd.DataFrame(r2_comparison).sort_values(['Target','GridSize'])\
    .to_excel(os.path.join(grid_folder, 'ANN_R2_Comparison_Train_vs_Test.xlsx'), index=False)
pd.DataFrame(pdp_records).to_pickle(os.path.join(grid_folder, 'pdp_records_ann.pkl'))

print("✅ ANN 训练与分析结果已保存。")


Fitting 20 folds for each of 100 candidates, totalling 2000 fits
